# Chain Ladder Reserving: Ultimate Claims by Line
This notebook loads the monthly claims into [chainladder-python](https://chainladder-python.readthedocs.io/) and estimates ultimate claims for buildings, contents and motor. It starts with claims-only methods (basic chain ladder, Mack, bootstrap ODP), then moves to methods that also use earned premium (Bornhuetter-Ferguson, Benktander, Cape Cod). Each step has charts and a short narrative. GitHub issue: #16.

**Key data limitation.** We only have one snapshot of each claim, valued at 31 Dec 2024. We have no payment or incurred history. So the triangles are **reporting triangles**: origin = loss month, development = months from loss month to report month, value = the claim's *current* (31 Dec 2024) count or net incurred. The development therefore measures **late reporting (pure IBNR)** only. It does not capture IBNER (movement on open claims already reported). We treat that separately at the end.

In [0]:
%pip install chainladder==0.10.1 duckdb==1.5.6 --quiet
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import chainladder as cl
import warnings
warnings.filterwarnings("ignore")

print("chainladder", cl.__version__)
LINES = ["buildings", "contents", "motor"]
VALUATION = pd.Timestamp("2024-12-31")

claims = pd.read_parquet("../data/claims.parquet")
claims["loss_date"] = pd.to_datetime(claims["loss_date"])
claims["report_date"] = pd.to_datetime(claims["report_date"])

# Net incurred as used in the target: incurred less recoveries, repudiated claims excluded
claims["net_incurred"] = np.where(claims["status"] == "repudiated", 0.0,
                                  claims["incurred_to_date"] - claims["recoveries"])
claims["claim_count"] = (claims["status"] != "repudiated").astype(int)
claims["loss_month"] = claims["loss_date"].dt.to_period("M").dt.to_timestamp()
claims["report_month"] = claims["report_date"].dt.to_period("M").dt.to_timestamp()

# Policies start on 2021-01-01, so the Dec-2020 loss month has no premium and is only a partial month: drop it
print("Dec-2020 losses dropped:", (claims["loss_month"] < "2021-01-01").sum())
claims = claims[claims["loss_month"] >= "2021-01-01"].copy()

summary = claims.groupby("line").agg(claims=("claim_count", "sum"), net_incurred=("net_incurred", "sum"),
                                     first_loss=("loss_date", "min"), last_report=("report_date", "max"))
display(summary)

claims.head()


## Step 1: Build monthly reporting triangles
Each claim is placed in a cell by **loss month** (origin) and **report month** (development). The value is its current claim count or net incurred. `cl.Triangle` turns the incremental data into cumulative triangles, with one triangle per line (`index="line"`) and two value columns (`claim_count`, `net_incurred`).

The check below confirms that the latest diagonal of each triangle adds up to the raw claim totals, so nothing is lost in the conversion.

In [0]:
incr = (claims.groupby(["line", "loss_month", "report_month"])[["claim_count", "net_incurred"]]
              .sum().reset_index())

tri = cl.Triangle(incr, origin="loss_month", development="report_month", index="line",
                  columns=["claim_count", "net_incurred"], cumulative=False,
                  origin_format="%Y-%m-%d", development_format="%Y-%m-%d").incr_to_cum()
print(tri)
print("Valuation date:", tri.valuation_date.date(), "| grain:", tri.origin_grain, tri.development_grain)

# Reconciliation: latest diagonal = raw totals
def tsum(t):
    """Sum of all cells in a single-index, single-column triangle."""
    return float(np.nansum(t.values))

check = pd.DataFrame({
    "triangle_count": [tsum(tri.loc[l]["claim_count"].latest_diagonal) for l in LINES],
    "raw_count": claims.groupby("line")["claim_count"].sum().reindex(LINES).values,
    "triangle_net": [tsum(tri.loc[l]["net_incurred"].latest_diagonal) for l in LINES],
    "raw_net": claims.groupby("line")["net_incurred"].sum().reindex(LINES).values}, index=LINES)
display(check.round(0))
assert np.allclose(check["triangle_net"], check["raw_net"]) and np.allclose(check["triangle_count"], check["raw_count"])

The triangles are 48 loss months × 48 development months (Jan 2021 to Dec 2024), for 3 lines and 2 measures. The latest diagonal reconciles exactly to the raw claims file (after dropping 274 Dec-2020 losses, which have no premium and are only a partial month).

## Step 2: Look at the development
The chart below groups the monthly origins into **accident years** (`tri.grain("OYDM")`) and plots cumulative reported claims by months of development. Two things to look for:
* **How fast claims are reported.** A curve that flattens early means little IBNR.
* **How stable the pattern is between years.** If the curves have different shapes, a single set of development factors may not fit all years.

In [0]:
tri_ay = tri.grain("OYDM")   # accident-year origins, monthly development

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for j, col in enumerate(["claim_count", "net_incurred"]):
    for i, line in enumerate(LINES):
        ax = axes[j, i]
        df = tri_ay.loc[line][col].to_frame()          # rows = accident year, cols = dev month
        for ay, row in df.iterrows():
            ax.plot(row.index.astype(int), row.values, marker=".", label=str(ay.year))
        ax.set_title(f"{line}: cumulative {col}")
        ax.set_xlabel("Development month"); ax.set_xlim(0, 48)
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax.legend(title="Accident year", fontsize=8)
plt.tight_layout(); plt.show()

# Share of each claim measure reported by development month (all origins pooled, ages with full data)
lag = (claims["report_month"].dt.year - claims["loss_month"].dt.year) * 12 + (claims["report_month"].dt.month - claims["loss_month"].dt.month)
mature = claims[claims["loss_month"] < "2023-01-01"].assign(lag=lag)     # origins with 24+ months of development
pct = {}
for col in ["claim_count", "net_incurred"]:
    s = mature.groupby(["line", "lag"])[col].sum().unstack(0).fillna(0).cumsum()
    pct[col] = (s / s.iloc[-1] * 100).loc[[0, 1, 2, 3, 6, 12, 18, 24]]
display(pd.concat(pct, axis=1).round(1).rename_axis("dev month (0 = same month)"))

**What the development shows** (table above, origins with 24+ months of development):
* **Contents** is reported fastest: about 97% of both count and cost is known within one month of the loss month.
* **Buildings** is next: about 93% of count and 89% of cost after one month, and 99% of both after 12 months.
* **Motor cost** develops much more slowly than motor count. Only 38% of the eventual cost is reported in the loss month, against 58% of the count. It is 85% after 6 months and 93% after 12 months. So the large motor claims (probably third-party injury and property) are reported late. This is where most of the IBNR will be.
* The accident-year curves have similar shapes from year to year, so a single set of factors per line is reasonable. (The first 12 months of each accident-year curve also include the year's exposure being earned, which is normal on an accident-year basis.)

## Step 3: Basic chain ladder (claims only)
We fit volume-weighted age-to-age factors on the **monthly** triangle with `cl.Development(average="volume")`. Then `cl.Chainladder()` projects every loss month to ultimate. No tail factor is used, because development ends within about 45 months and the oldest origin already has 47 months of development (`TailConstant(1.0)`).

The charts show:
1. the cumulative development factor (CDF) by age, shown as % of ultimate reported, and
2. IBNR by loss month, which should be concentrated in the last few months of 2024.

In [0]:
def by_line(model, col, name):
    """Total reported, ultimate and IBNR per line for one value column."""
    rows = {}
    for l in LINES:
        rep = tsum(tri.loc[l][col].latest_diagonal)
        ult = tsum(model.ultimate_.loc[l][col])
        rows[l] = {"reported": rep, "ultimate": ult, "ibnr": ult - rep, "ibnr_pct": (ult / rep - 1) * 100}
    return pd.DataFrame(rows).T.assign(method=name, measure=col)

def by_origin(model_tri, col):
    """Origin x line DataFrame from a model output triangle (e.g. ultimate_, ibnr_)."""
    return pd.concat({l: model_tri.loc[l][col].to_frame().iloc[:, 0] for l in LINES}, axis=1)

dev_vol = cl.Development(average="volume").fit(tri)
cl_basic = cl.Chainladder().fit(cl.TailConstant(1.0).fit_transform(dev_vol.transform(tri)))

res_basic = pd.concat([by_line(cl_basic, c, "Chainladder (volume, all)") for c in ["claim_count", "net_incurred"]])
display(res_basic.round(1))

# Chart 1: % of ultimate reported by development age = 1 / CDF
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, col in zip(axes, ["claim_count", "net_incurred"]):
    for l in LINES:
        cdf = cl_basic.cdf_.loc[l][col].to_frame().iloc[0]
        ax.plot(range(1, len(cdf) + 1), 100 / cdf.values, marker=".", label=l)
    ax.set_title(f"% of ultimate reported by development month ({col})")
    ax.set_xlabel("Development month (1 = loss month)"); ax.set_ylabel("% reported"); ax.set_xlim(0, 36)
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

# Chart 2: IBNR by loss month (last 24 months)
ibnr_m = by_origin(cl_basic.ibnr_, "net_incurred")
ax = ibnr_m.iloc[-24:].plot(kind="bar", stacked=True, figsize=(16, 4.5), width=.85)
ax.set_xticklabels([d.strftime("%Y-%m") for d in ibnr_m.index[-24:]], rotation=90)
ax.set_title("Chain ladder net incurred IBNR by loss month (last 24 months)"); ax.set_ylabel("CU")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout(); plt.show()

**Basic chain ladder results.** IBNR is small for the property lines. For contents it is about 1% of reported (count and cost). For buildings it is about 2% of count and 2.4% of cost. Motor is different: IBNR is about 2% of claim count but **7.3% of cost (about 31.6m CU)**, because the late-reported motor claims are much larger than average. Nearly all IBNR sits in the last 3-6 loss months of 2024, as expected.

## Step 4: How sensitive is this to the averaging method?
Monthly link ratios at older ages are based on few claims, so the choice of averaging matters. We re-fit with:
* **volume-weighted, all periods**: the base case above
* **simple average, all periods**: gives each origin month equal weight
* **volume-weighted, latest 12 diagonals**: reflects the most recent reporting speed
* **regression average**: fits each age's factor by regression
* **volume-weighted, excluding the highest and lowest factor at each age**: removes outliers

The chart compares total net incurred IBNR by line under each option.

In [0]:
dev_options = {
    "Volume, all": cl.Development(average="volume"),
    "Simple, all": cl.Development(average="simple"),
    "Volume, last 12": cl.Development(average="volume", n_periods=12),
    "Regression, all": cl.Development(average="regression"),
    "Volume, excl hi/lo": cl.Development(average="volume", drop_high=True, drop_low=True),
}
sens = []
for name, dev in dev_options.items():
    m = cl.Chainladder().fit(dev.fit_transform(tri))
    sens.append(pd.concat([by_line(m, c, name) for c in ["claim_count", "net_incurred"]]))
sens = pd.concat(sens)

piv = sens.reset_index().pivot_table(index=["measure", "index"], columns="method", values="ibnr_pct")[list(dev_options)]
display(piv.round(2).rename_axis(["measure", "line"]))

ax = (sens[sens.measure == "net_incurred"].reset_index()
      .pivot(index="index", columns="method", values="ibnr")[list(dev_options)]
      .plot(kind="bar", figsize=(14, 4.5), width=.8))
ax.set_title("Net incurred IBNR by averaging method"); ax.set_xlabel(""); ax.set_ylabel("CU")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}")); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## Step 5: Mack chain ladder: how uncertain is the IBNR?
`cl.MackChainladder` gives the same central estimate as the volume-weighted chain ladder. It also estimates the **standard error** of each origin's ultimate (process risk plus parameter risk), using the variability of the link ratios. We show the total IBNR, its standard error and coefficient of variation (CV) by line. The chart shows 2024 monthly ultimates with ±2 standard error bands.

In [0]:
# Fit one Mack model per line and measure so the total (all-origin) standard error is easy to read
mack_rows, mack_fits = [], {}
for l in LINES:
    for c in ["claim_count", "net_incurred"]:
        sub = tri.loc[l][c]
        m = cl.MackChainladder().fit(cl.Development(average="volume").fit_transform(sub))
        mack_fits[(l, c)] = m
        ibnr = tsum(m.ibnr_)
        se = float(np.asarray(m.total_mack_std_err_).ravel()[0])
        mack_rows.append({"line": l, "measure": c, "ibnr": ibnr, "total_std_err": se, "cv_of_ibnr": se / ibnr})
mack_tbl = pd.DataFrame(mack_rows).set_index(["measure", "line"])
display(mack_tbl.round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, l in zip(axes, LINES):
    m = mack_fits[(l, "net_incurred")]
    ult = m.ultimate_.to_frame().iloc[:, 0]
    se = m.mack_std_err_.to_frame().iloc[:, -1]
    rep = tri.loc[l]["net_incurred"].latest_diagonal.to_frame().iloc[:, 0]
    idx = ult.index >= "2024-01-01"
    x = np.arange(idx.sum())
    ax.bar(x, rep[idx], color="lightgrey", label="Reported")
    ax.errorbar(x, ult[idx], yerr=2 * se[idx], fmt="o", color="C3", capsize=3, label="Ultimate ±2 SE")
    ax.set_xticks(x); ax.set_xticklabels([d.strftime("%b") for d in ult.index[idx]])
    ax.set_title(f"{l}: 2024 net incurred, Mack ultimate ±2 SE")
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}")); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Averaging sensitivity.** Counts hardly move: buildings IBNR is 1.8-1.9% of reported, contents about 1.0% and motor about 2.0% under every option. Buildings and contents cost also stay close to the base case. **Motor cost is the sensitive one.** Its IBNR is 7.3-7.5% of reported with all-period averages, but about 5.9% with the latest 12 diagonals or with high/low factors excluded. That points to a few large, late-reported motor claims in some older months, and possibly faster reporting recently.

**Mack.** The Mack IBNR matches the basic chain ladder exactly, which is a useful cross-check. The uncertainty is modest for counts (CV 3-8%). It is larger for cost: CV 13% for buildings, 17% for contents and 21% for motor. For motor, ±2 SE is roughly ±13m CU around an IBNR of 31.6m.

## Step 6: Bootstrap ODP: a full distribution of IBNR
`cl.BootstrapODPSample` resamples the scaled Pearson residuals of an over-dispersed Poisson model, which gives 1,000 simulated net incurred triangles per line. Each simulated triangle is then projected with a chain ladder. This gives a distribution of IBNR without assuming normality. We report the mean and the 75th / 95th / 99.5th percentiles. The 99.5th is the Solvency II reserve-risk percentile.

In [0]:
N_SIMS = 1000
boot_ibnr, boot_rows = {}, []
for l in LINES:
    sub = tri.loc[l]["net_incurred"]
    sims = cl.BootstrapODPSample(n_sims=N_SIMS, random_state=42).fit_transform(sub)
    m = cl.Chainladder().fit(cl.Development(average="volume").fit_transform(sims))
    ibnr = np.nansum(m.ibnr_.values, axis=(1, 2, 3))          # one total IBNR per simulation
    boot_ibnr[l] = ibnr
    boot_rows.append({"line": l, "chainladder_ibnr": tsum(cl_basic.ibnr_.loc[l]["net_incurred"]),
                      "boot_mean": ibnr.mean(), "boot_sd": ibnr.std(),
                      "p75": np.percentile(ibnr, 75), "p95": np.percentile(ibnr, 95), "p99.5": np.percentile(ibnr, 99.5)})
boot_tbl = pd.DataFrame(boot_rows).set_index("line")
display(boot_tbl.round(0))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, l in zip(axes, LINES):
    ax.hist(boot_ibnr[l], bins=40, color="C0", alpha=.7)
    for q, ls in [("chainladder_ibnr", "-"), ("p95", "--"), ("p99.5", ":")]:
        ax.axvline(boot_tbl.loc[l, q], color="C3", ls=ls, label=q)
    ax.set_title(f"{l}: bootstrap ODP net incurred IBNR ({N_SIMS} sims)")
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1e6:,.1f}m")); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Bootstrap ODP.** The bootstrap means are within 0.5% of the chain ladder IBNR for all three lines, which confirms the central estimate. The 99.5th percentiles are about 1.5× the mean for buildings (8.7m vs 5.9m), 1.4× for contents (1.0m vs 0.7m) and 1.3× for motor (42.3m vs 31.5m). The ODP spread for motor (SD about 3.8m) is narrower than Mack (6.6m). The ODP model assumes one shared variance scale across all ages, so it gives less weight to the volatile late large motor claims. Mack is the more prudent view here.

## Step 7: Earned premium and exposure by loss month
The premium-based methods need **earned premium for each loss month**. Each policy row earns `written_premium / 365` per day (and `1/365` of a policy-year of exposure per day) from `period_start` to `period_end`. This matches the `exposure_standardised × written_premium` definition in `home_insurance_model` (days / 365). We spread this over calendar days with a start/stop step function, then sum by month.

Check: total monthly earned premium must equal Σ (days / 365 × written_premium) over all policy rows.

In [0]:
days = pd.date_range("2021-01-01", "2025-01-01", freq="D")       # one extra day for the stop events
earned = {}
for l in LINES:
    p = pd.read_parquet(f"../data/policies_{l}.parquet", columns=["period_start", "period_end", "written_premium"])
    s = (pd.to_datetime(p["period_start"]) - days[0]).dt.days.values
    e = (pd.to_datetime(p["period_end"]) - days[0]).dt.days.values + 1       # inclusive end -> stop the day after
    rate_p = np.zeros(len(days)); rate_e = np.zeros(len(days))
    np.add.at(rate_p, s, p["written_premium"].values / 365); np.add.at(rate_p, e, -p["written_premium"].values / 365)
    np.add.at(rate_e, s, 1 / 365); np.add.at(rate_e, e, -1 / 365)
    daily = pd.DataFrame({"earned_premium": rate_p.cumsum(), "earned_exposure": rate_e.cumsum()}, index=days)[:-1]
    m = daily.resample("MS").sum()
    expected = ((e - s) / 365 * p["written_premium"].values).sum()
    print(f"{l:9s} earned premium {m['earned_premium'].sum():,.0f}  vs  check {expected:,.0f}  | policy-years {m['earned_exposure'].sum():,.0f}")
    assert np.isclose(m["earned_premium"].sum(), expected, rtol=1e-9)
    earned[l] = m
    del p

earned_df = pd.concat(earned, names=["line", "loss_month"]).reset_index()
earned_df["valuation"] = VALUATION
# Exposure "triangle": origin = loss month, single development = valuation date
prem = cl.Triangle(earned_df, origin="loss_month", development="valuation", index="line",
                   columns=["earned_premium", "earned_exposure"], cumulative=True)

# Reported loss ratio and frequency by month, as context
ep_m = pd.concat({l: earned[l]["earned_premium"] for l in LINES}, axis=1)
rep_m = pd.concat({l: tri.loc[l]["net_incurred"].latest_diagonal.to_frame().iloc[:, 0] for l in LINES}, axis=1)
ep_m.index = rep_m.index
fig, axes = plt.subplots(1, 2, figsize=(18, 4.5))
ep_m.plot(kind="bar", ax=axes[0], width=.85)
axes[0].set_title("Earned premium by loss month"); axes[0].set_xticklabels([d.strftime("%Y-%m") for d in ep_m.index], rotation=90, fontsize=7)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1e6:,.1f}m"))
(rep_m / ep_m * 100).plot(ax=axes[1], marker=".")
axes[1].set_title("Reported (undeveloped) net incurred loss ratio by loss month, %"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

Earned premium reconciles exactly to the policy-level check for every line: 191.0m buildings, 80.2m contents and 438.3m motor over 2021-24. Premium and exposure grow steadily over the four years. Even before development, the reported loss ratios are high: buildings is about **130%**, motor about 98% and contents about 82%. The recent months dip only because they are not yet fully reported, and those are the months the premium methods help with.

## Step 8: Methods that use earned premium (and exposure)
The chain ladder relies entirely on the few claims reported so far in the most recent months, which is volatile, especially for motor. The premium-based methods mix in an **a priori expected loss ratio (ELR)** instead:
* **Bornhuetter-Ferguson (BF):** ultimate = reported + ELR × premium × (1 − 1/CDF). The IBNR depends only on premium and the reporting pattern, not on what has been reported so far.
* **Benktander (2 iterations):** BF applied twice. It sits between BF and chain ladder, giving more weight to the claims experience as an origin matures.
* **Cape Cod:** estimates the ELR from the data itself: Σ reported / Σ (premium × % reported). With `decay=1` all origins share one ELR. With `decay=0.75` (generalised Cape Cod), nearby origins get more weight.

The BF / Benktander prior ELR is each line's chain ladder ultimate loss ratio for 2021-23. Those years are almost fully reported, so this ELR does not depend on the immature 2024 months. For **claim counts** we use the same methods with **earned exposure (policy-years)** as the weight, so the prior is a claim frequency.

In [0]:
pairs = {"net_incurred": "earned_premium", "claim_count": "earned_exposure"}
prem_rows, prem_origin, elr_rows = [], {}, []
for l in LINES:
    for col, wcol in pairs.items():
        X = cl.Development(average="volume").fit_transform(tri.loc[l][col])
        w = prem.loc[l][wcol]
        w_s = w.latest_diagonal.to_frame().iloc[:, 0]
        cl_ult = cl_basic.ultimate_.loc[l][col].to_frame().iloc[:, 0]
        w_s.index = cl_ult.index
        mature_idx = cl_ult.index < "2024-01-01"
        elr = cl_ult[mature_idx].sum() / w_s[mature_idx].sum()       # prior from 2021-23 chain ladder

        models = {
            "Chainladder": cl.Chainladder().fit(X),
            "BF": cl.BornhuetterFerguson(apriori=elr).fit(X, sample_weight=w),
            "Benktander (2 iter)": cl.Benktander(apriori=elr, n_iters=2).fit(X, sample_weight=w),
            "Cape Cod (decay 1)": cl.CapeCod(decay=1.0, trend=0.0).fit(X, sample_weight=w),
            "Cape Cod (decay 0.75)": cl.CapeCod(decay=0.75, trend=0.0).fit(X, sample_weight=w),
        }
        rep = tsum(tri.loc[l][col].latest_diagonal)
        cc_elr = float(np.nanmean(models["Cape Cod (decay 1)"].apriori_.values))
        elr_rows.append({"line": l, "measure": col, "prior_ELR_2021_23": elr, "capecod_ELR": cc_elr})
        for name, m in models.items():
            ult = tsum(m.ultimate_)
            prem_rows.append({"line": l, "measure": col, "method": name, "reported": rep, "ultimate": ult,
                              "ibnr": ult - rep, "ibnr_pct": (ult / rep - 1) * 100})
            prem_origin[(l, col, name)] = m.ultimate_.to_frame().iloc[:, 0]

elr_tbl = pd.DataFrame(elr_rows).set_index(["measure", "line"])
print("Prior ELR (net incurred / earned premium) and frequency (claims / policy-year):")
display(elr_tbl.round(4))
prem_res = pd.DataFrame(prem_rows)
display(prem_res.pivot_table(index=["measure", "line"], columns="method", values="ibnr_pct")
        [list(models)].round(2))

# Chart: 2024 monthly ultimate loss ratio by method
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, l in zip(axes, LINES):
    w_s = prem.loc[l]["earned_premium"].latest_diagonal.to_frame().iloc[:, 0].values
    for name in models:
        u = prem_origin[(l, "net_incurred", name)]
        lr = u.values / w_s * 100
        idx = u.index >= "2024-01-01"
        ax.plot(u.index[idx].strftime("%b"), lr[idx], marker="o", label=name)
    ax.axhline(elr_tbl.loc[("net_incurred", l), "prior_ELR_2021_23"] * 100, color="grey", ls=":", label="Prior ELR")
    ax.set_title(f"{l}: 2024 ultimate loss ratio by method, %"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Premium-based results.** The 2021-23 prior loss ratios are buildings **130%**, contents **83%** and motor **107%**. The data-driven Cape Cod ELRs (133%, 83%, 105%) are very close, so the prior is consistent with the claims experience. Prior frequencies are 4.5%, 4.0% and 14.5% claims per policy-year.
* **Buildings and contents:** BF and Cape Cod give slightly *less* IBNR than chain ladder (buildings cost 2.2% vs 2.4%). The last few 2024 months were reported a little heavier than usual, and the chain ladder carries that forward. BF gives it less weight.
* **Motor:** every method gives about 7.2-7.4% of reported cost. So the motor IBNR of about 31m is robust to the choice of method. The averaging period (Step 4) matters more than the method.
* Benktander lands between chain ladder and BF, as expected.

## Step 9: Compare all methods and select
The table and chart bring together net incurred IBNR from every method. As an independent check, we recompute BF by hand (reported + ELR × EP × (1 − 1/CDF)) and confirm it matches chainladder's BF.

In [0]:
net = "net_incurred"
comp = {}
for name in dev_options:                                              # claims-only: chain ladder averaging options
    comp[f"CL {name}"] = sens[(sens.measure == net) & (sens.method == name)]["ibnr"].reindex(LINES)
comp["Mack"] = mack_tbl.xs(net)["ibnr"].reindex(LINES)
comp["Bootstrap ODP mean"] = boot_tbl["boot_mean"].reindex(LINES)
for name in ["BF", "Benktander (2 iter)", "Cape Cod (decay 1)", "Cape Cod (decay 0.75)"]:
    comp[name] = prem_res[(prem_res.measure == net) & (prem_res.method == name)].set_index("line")["ibnr"].reindex(LINES)
comp = pd.DataFrame(comp).T
comp["total"] = comp.sum(axis=1)
display((comp / 1e6).round(2).rename_axis("Net incurred IBNR (m CU)"))

ax = (comp[LINES] / 1e6).plot(kind="barh", stacked=True, figsize=(12, 5))
ax.set_title("Net incurred IBNR by method (m CU)"); ax.set_xlabel("m CU"); ax.invert_yaxis(); ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

# Independent check of BF: reported + ELR * EP * (1 - 1/CDF), from first principles
for l in LINES:
    rep = tri.loc[l][net].latest_diagonal.to_frame().iloc[:, 0].values
    ep = prem.loc[l]["earned_premium"].latest_diagonal.to_frame().iloc[:, 0].values
    cdf_age = cl_basic.cdf_.loc[l][net].to_frame().iloc[0].values          # CDF by age 1..48
    ages = np.arange(len(rep))[::-1]                                     # newest origin is age 1 (index 0)
    cdf_origin = np.append(cdf_age, 1.0)[ages]
    bf_manual = (rep + elr_tbl.loc[(net, l), "prior_ELR_2021_23"] * ep * (1 - 1 / cdf_origin)).sum() - rep.sum()
    bf_lib = comp.loc["BF", l]
    print(f"{l:9s} BF IBNR manual {bf_manual:,.0f} | chainladder {bf_lib:,.0f} | diff {bf_manual/bf_lib-1:+.4%}")

**Comparison.** The hand-calculated BF matches chainladder's BF to the unit for all three lines. Total net incurred IBNR is **37-39m CU** for 9 of the 11 estimates. The only exceptions are the two "recent-weighted" chain ladders (latest 12 diagonals, and excluding high/low), at 31-32m. The whole difference is motor (25m vs 31m). Buildings stays in a 5.2-6.2m range and contents 0.6-0.7m.

**Selection.** We take **Benktander (2 iterations)** as the central estimate for every line: about 5.7m buildings, 0.7m contents and 31.4m motor. It uses each origin's own reporting where that is credible (older months) and the premium-based prior where it is not (the last few months of 2024). It also sits in the middle of the range for every line. The recent-weighted motor chain ladders are a plausible downside, which the team should discuss.

## Step 10: Development factors by accident year, for the modelling target
The main use of this work in the competition is to **develop the training claims to ultimate** (issue #16). The table gives, for each line and accident year, the factor *selected ultimate / reported net incurred* and the developed loss ratio. The open-claim share is shown as a pointer to IBNER, which these reporting triangles cannot measure.

In [0]:
SELECTED = "Benktander (2 iter)"
ay_rows = []
for l in LINES:
    ult = prem_origin[(l, net, SELECTED)]
    rep = tri.loc[l][net].latest_diagonal.to_frame().iloc[:, 0]; rep.index = ult.index
    ep = ep_m[l]; ep.index = ult.index
    cnt_ult = prem_origin[(l, "claim_count", SELECTED)]
    cnt_rep = tri.loc[l]["claim_count"].latest_diagonal.to_frame().iloc[:, 0]; cnt_rep.index = ult.index
    df = pd.DataFrame({"reported": rep, "ultimate": ult, "ep": ep, "cnt_rep": cnt_rep, "cnt_ult": cnt_ult})
    g = df.groupby(df.index.year).sum()
    sub = claims[(claims.line == l) & (claims.status != "repudiated")]
    open_share = sub.groupby(sub.loss_date.dt.year).apply(lambda d: (d.status == "open").mean() * 100)
    for ay, r in g.iterrows():
        ay_rows.append({"line": l, "accident_year": ay, "reported_net": r.reported, "ultimate_net": r.ultimate,
                        "cost_dev_factor": r.ultimate / r.reported, "count_dev_factor": r.cnt_ult / r.cnt_rep,
                        "reported_LR_%": r.reported / r.ep * 100, "ultimate_LR_%": r.ultimate / r.ep * 100,
                        "open_claims_%": open_share.get(ay, np.nan)})
ay_tbl = pd.DataFrame(ay_rows).set_index(["line", "accident_year"])
display(ay_tbl.round({"reported_net": 0, "ultimate_net": 0, "cost_dev_factor": 4, "count_dev_factor": 4,
                      "reported_LR_%": 1, "ultimate_LR_%": 1, "open_claims_%": 1}))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, l in zip(axes, LINES):
    d = ay_tbl.loc[l]
    x = np.arange(len(d))
    ax.bar(x - .2, d["reported_LR_%"], .4, label="Reported LR", color="lightgrey")
    ax.bar(x + .2, d["ultimate_LR_%"], .4, label=f"Ultimate LR ({SELECTED})", color="C0")
    for xi, f in zip(x, d["cost_dev_factor"]):
        ax.text(xi + .2, d["ultimate_LR_%"].iloc[xi] + 1, f"×{f:.3f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(d.index); ax.set_title(f"{l}: net incurred loss ratio by accident year, %")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## Summary
**Development factors (selected: Benktander, net incurred, ultimate / reported):**

| Line | AY 2021 | AY 2022 | AY 2023 | AY 2024 | Ultimate LR 2021-24 |
| --- | --- | --- | --- | --- | --- |
| Buildings | 1.000 | 1.000 | 1.004 | **1.081** | 129-139% |
| Contents | 1.000 | 1.000 | 1.002 | **1.037** | 81-85% |
| Motor | 1.003 | 1.007 | 1.032 | **1.293** | 101-108% |

* **IBNR is concentrated in 2024 and in motor.** Motor 2024 reported cost needs about +29% for late-reported claims, mainly large ones (the count factor is only 1.08). Buildings 2024 needs +8% and contents +4%. Years 2021-22 are essentially fully reported.
* **The methods agree.** Chain ladder, Mack, bootstrap ODP, BF, Benktander and Cape Cod all give 37-39m CU total IBNR. The exception is the recent-weighted motor chain ladders (about 25m). For motor, Mack's ±2 standard error is about ±13m.
* **Once developed, the 2024 dip in loss ratios goes away** (the "no falling trend" check asked for in #16). Buildings is about 130% in 2021-23 and rises to 139% in 2024, so it is under-priced and getting worse. Contents is flat at 81-85%. Motor 2024 develops to 101%, still a little below 105-108% in 2021-23. That small gap may be IBNER on open motor claims, or genuine improvement.
* **IBNER is not included.** 25% of buildings and 21% of motor 2024 claims are still open, and their case estimates may still move. With only one valuation date we cannot estimate that from triangles. Options: (a) accept case reserves as best estimate, or (b) compare open vs closed severities by age as a proxy.

**Next steps:**
* Apply the accident-year (or loss-month) cost factors to `net_incurred` in the claims aggregation (#15) before fitting the GLMs, so the 2024 holdout and refits target ultimate cost.
* Use the developed 2024 loss ratios when setting the v00x level loadings, in place of the separate large-loss and development guesses.

## Step 11: Shared claims database (#15)
This step builds the per-policy-row claims data from #15 and saves it to a **DuckDB** file in the repo, `data/claims.duckdb`, so other notebooks can query it without re-running this one.

**Matching.** Each claim is matched to the policy row (`row_id`) with the same `policy_id` whose `period_start` to `period_end` contains the `loss_date`. We check that no claim matches two rows and report any claims that don't match.

**Development.** Each non-repudiated claim's net incurred is multiplied by the Benktander cost factor for its line and loss month (ultimate / reported, from Step 8). Losses before 2021 get a factor of 1. This gives `net_incurred_dev`. The factors only scale up rows that *already* have claims. Claims not yet reported can't be attached to a row, so a model trained on `net_incurred_dev` puts the IBNR into severity rather than frequency. The monthly factors and IBNR are saved too (`dev_factors`), so you can allocate IBNR by exposure instead if you prefer.

**Tables in `data/claims.duckdb`:**

| Table | Grain | Contents |
| --- | --- | --- |
| `claims` | one row per claim | raw claim fields + `row_id` (null if unmatched), `loss_month`, `net_incurred`, `cost_dev_factor`, `net_incurred_dev` |
| `claims_by_row` | one row per policy row with ≥1 matched claim | counts by status, `total_paid`, `total_incurred`, `total_recoveries`, `net_incurred` (target, undeveloped), `net_incurred_dev` |
| `dev_factors` | line × loss month | reported, Benktander ultimate, IBNR, cost and count factors, earned premium |

Policy rows with no claims are **not** stored. Left-join `claims_by_row` onto the policy file and fill with 0.

In [0]:
# --- Monthly development factors (Benktander, Step 8) ---
fac_rows = []
for l in LINES:
    for col, wcol in pairs.items():
        u = prem_origin[(l, col, SELECTED)]
        r = tri.loc[l][col].latest_diagonal.to_frame().iloc[:, 0]; r.index = u.index
        fac_rows.append(pd.DataFrame({"line": l, "loss_month": u.index, "measure": col, "reported": r.values, "ultimate": u.values}))
fac = pd.concat(fac_rows).pivot_table(index=["line", "loss_month"], columns="measure", values=["reported", "ultimate"])
fac.columns = [f"{a}_{b}" for a, b in fac.columns]
dev_factors = fac.reset_index()
dev_factors["cost_dev_factor"] = dev_factors["ultimate_net_incurred"] / dev_factors["reported_net_incurred"]
dev_factors["count_dev_factor"] = dev_factors["ultimate_claim_count"] / dev_factors["reported_claim_count"]
dev_factors["ibnr_net_incurred"] = dev_factors["ultimate_net_incurred"] - dev_factors["reported_net_incurred"]
dev_factors = dev_factors.merge(earned_df[["line", "loss_month", "earned_premium", "earned_exposure"]], on=["line", "loss_month"])
dev_factors["method"] = SELECTED

# --- Claim level: match to policy row ---
clm = pd.read_parquet("../data/claims.parquet")
clm["loss_date"] = pd.to_datetime(clm["loss_date"]); clm["report_date"] = pd.to_datetime(clm["report_date"])
clm["loss_month"] = clm["loss_date"].dt.to_period("M").dt.to_timestamp()
clm["repudiated"] = clm["status"] == "repudiated"
clm["net_incurred"] = np.where(clm["repudiated"], 0.0, clm["incurred_to_date"] - clm["recoveries"])

pol = pd.concat([pd.read_parquet(f"../data/policies_{l}.parquet", columns=["row_id", "policy_id", "period_start", "period_end"])
                 .assign(line=l) for l in LINES])
pol["period_start"] = pd.to_datetime(pol["period_start"]); pol["period_end"] = pd.to_datetime(pol["period_end"])
pol = pol[pol["policy_id"].isin(clm["policy_id"])]
m = clm[["claim_id", "policy_id", "loss_date"]].merge(pol, on="policy_id")
m = m[(m["loss_date"] >= m["period_start"]) & (m["loss_date"] <= m["period_end"])]
n_multi = m["claim_id"].duplicated().sum()
assert n_multi == 0, f"{n_multi} claims matched to more than one row"
clm = clm.merge(m[["claim_id", "row_id", "period_start", "period_end"]], on="claim_id", how="left")

# --- Apply development factors ---
clm = clm.merge(dev_factors[["line", "loss_month", "cost_dev_factor"]], on=["line", "loss_month"], how="left")
clm["cost_dev_factor"] = clm["cost_dev_factor"].fillna(1.0)          # pre-2021 losses
clm["net_incurred_dev"] = clm["net_incurred"] * clm["cost_dev_factor"]

unm = clm[clm["row_id"].isna()]
print(f"Claims: {len(clm):,} | matched to a row: {clm.row_id.notna().sum():,} | unmatched: {len(unm):,} "
      f"({unm.net_incurred.sum():,.0f} CU net incurred)")
display(unm.groupby(["line", unm.loss_date.dt.year.rename("loss_year")]).agg(claims=("claim_id", "size"), net_incurred=("net_incurred", "sum")))

# --- Aggregate per policy row ---
mc = clm[clm["row_id"].notna()].assign(
    is_claim=lambda d: (~d["repudiated"]).astype(int), is_open=lambda d: (d["status"] == "open").astype(int),
    is_closed=lambda d: (d["status"] == "closed").astype(int), is_rep=lambda d: d["repudiated"].astype(int))
claims_by_row = mc.groupby(["row_id", "policy_id", "line", "period_start", "period_end"]).agg(
    claim_count=("is_claim", "sum"),
    n_open=("is_open", "sum"),
    n_closed=("is_closed", "sum"),
    n_repudiated=("is_rep", "sum"),
    total_paid=("paid_to_date", "sum"),
    total_incurred=("incurred_to_date", "sum"),
    total_recoveries=("recoveries", "sum"),
    net_incurred=("net_incurred", "sum"),
    net_incurred_dev=("net_incurred_dev", "sum"),
).reset_index()

# Sense checks: per-row totals reconcile to matched claims; developed total = reported + allocated IBNR
assert np.isclose(claims_by_row["net_incurred"].sum(), mc["net_incurred"].sum())
assert claims_by_row["row_id"].is_unique
print(f"Policy rows with claims: {len(claims_by_row):,}")
chk = claims_by_row.groupby("line")[["claim_count", "net_incurred", "net_incurred_dev"]].sum()
chk["uplift_%"] = (chk["net_incurred_dev"] / chk["net_incurred"] - 1) * 100
display(chk.round(1))
display(claims_by_row.head())

In [0]:
import duckdb, os, shutil, tempfile

DB_PATH = os.path.abspath("../data/claims.duckdb")
# Build in local temp storage first (DuckDB needs random-access writes), then copy into the repo
tmp_db = os.path.join(tempfile.mkdtemp(), "claims.duckdb")
claims_out = clm.drop(columns=["repudiated"])
with duckdb.connect(tmp_db) as con:
    for name, df in {"claims": claims_out, "claims_by_row": claims_by_row, "dev_factors": dev_factors}.items():
        con.register("df_view", df)
        con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM df_view")
        con.unregister("df_view")
    con.execute("COMMENT ON TABLE claims IS 'One row per claim, matched to policy row_id by policy_id + loss_date in period. net_incurred excludes repudiated. net_incurred_dev = net_incurred x Benktander cost factor (workings/chainladder, #15/#16)'")
    con.execute("COMMENT ON TABLE claims_by_row IS 'Claims aggregated per policy row_id (rows with >=1 matched claim only; left-join and fill 0). Valued 31 Dec 2024.'")
    con.execute("COMMENT ON TABLE dev_factors IS 'Benktander ultimate / reported by line and loss month, with earned premium (workings/chainladder)'")
    con.execute("CHECKPOINT")
shutil.copyfile(tmp_db, DB_PATH)
print(f"Wrote {DB_PATH} ({os.path.getsize(DB_PATH)/1e6:.1f} MB)")

**How to use it from another notebook** (in `workings/`, after `%pip install duckdb==1.5.6`):
* Connect read-only: `con = duckdb.connect("../data/claims.duckdb", read_only=True)`
* Pull a table: `con.sql("SELECT * FROM claims_by_row WHERE line = 'motor'").df()`
* Build the target: left-join `claims_by_row` onto `policies_<line>` on `row_id`, and fill missing `claim_count`, `net_incurred` and `net_incurred_dev` with 0.

The cell below reads the file back the same way and checks it against the in-memory data.

In [0]:
con = duckdb.connect(DB_PATH, read_only=True)
display(con.sql("SELECT table_name, estimated_size AS rows, comment FROM duckdb_tables() ORDER BY table_name").df())

rb = con.sql("""
    SELECT line, SUM(claim_count) AS claim_count, SUM(net_incurred) AS net_incurred, SUM(net_incurred_dev) AS net_incurred_dev
    FROM claims_by_row GROUP BY line ORDER BY line""").df().set_index("line")
display(rb.round(0))
assert np.allclose(rb[["claim_count", "net_incurred", "net_incurred_dev"]].values,
                   chk.loc[rb.index, ["claim_count", "net_incurred", "net_incurred_dev"]].values)

# Independent check against the raw parquet: matched + unmatched net incurred = raw total (repudiated excluded)
raw = pd.read_parquet("../data/claims.parquet")
raw_net = (raw["incurred_to_date"] - raw["recoveries"])[raw["status"] != "repudiated"].sum()
db_net = con.sql("SELECT SUM(net_incurred) FROM claims").fetchone()[0]
print(f"Raw parquet net incurred {raw_net:,.0f} | DuckDB claims table {db_net:,.0f}")
assert np.isclose(raw_net, db_net) and con.sql("SELECT COUNT(*) FROM claims").fetchone()[0] == len(raw)
con.close()